# SDP Medallion — declarative bronze → silver → gold

Build a real food-delivery medallion as **Delta tables in Unity Catalog** using
**Spark Declarative Pipelines**. You declare *what* each table is; SDP infers the
DAG, orders execution, and handles the writes.

**Why this matters — declarative vs imperative (same pipeline):**

| | lines | you manage |
|---|---|---|
| `transformations/*.py` (SDP) | ~3 short functions/layer | nothing but the SELECT |
| `imperative_medallion.py` | ~2x the code | session, exec order, every write |


### The declarative form (this is the whole bronze layer)
```python
@dp.materialized_view(name='bronze.orders',
    table_properties={'location': '.../orders', 'provider': 'delta'})
def orders():
    return spark.read.parquet(ORDERS_PATH).withColumn('event_timestamp', ...)
```
SDP sees `spark.read.table('bronze.orders')` in silver and wires the DAG itself.
Compare with `imperative_medallion.py` (open it side-by-side).

In [ ]:
# Clean slate so the pipeline can re-run. UC's connector has no TRUNCATE; the
# verified reset is a REST DELETE per table (mirrors demos/sdp-medallion/teardown.sh),
# here against Scott's remote UC with the token, namespaced by DEMO_NS.
import os, urllib.request
UC = os.environ.get('UC_URI', 'https://uc.openlakehousedemos.dev')
TOK = os.environ.get('UC_TOKEN', '')
NS = os.environ.get('DEMO_NS', '')
TABLES = [('bronze','orders'),('bronze','dim_locations'),
          ('silver','orders_enriched'),('silver','order_lifecycle'),
          ('gold','hourly_metrics'),('gold','delivery_performance'),('gold','brand_summary')]
for layer, t in TABLES:
    full = f'managed_demo.{NS}{layer}.{t}'
    req = urllib.request.Request(f'{UC}/api/2.1/unity-catalog/tables/{full}',
                                 method='DELETE', headers={'Authorization': f'Bearer {TOK}'})
    try:
        urllib.request.urlopen(req); print('dropped', full)
    except Exception as e:
        print('skip   ', full, getattr(e, 'code', e))  # 404 = already gone, fine
print('teardown complete')

In [ ]:
# Run the declarative pipeline. spark-pipelines builds the graph and materializes
# all 7 Delta tables into Unity Catalog, in dependency order.
import subprocess, os
demo = os.environ.get('SDP_DEMO_DIR', '/home/jovyan/demos/sdp-medallion')
print(subprocess.run(['spark-pipelines','run'], cwd=demo, text=True,
      capture_output=True).stdout[-2000:])

In [ ]:
import os
NS = os.environ.get('DEMO_NS', '')
spark.table(f'managed_demo.{NS}gold.brand_summary').orderBy('total_revenue', ascending=False).show(8, truncate=False)
spark.table(f'managed_demo.{NS}gold.delivery_performance').show(truncate=False)